## Import

In [0]:
from pathlib import Path
import requests
import zipfile as z
import logging
import datetime as dt

## Paths

In [0]:
VolumePath = "/Volumes/dev/landing/inmet"
inmetPath = "https://portal.inmet.gov.br/uploads/dadoshistoricos"

## Config

In [0]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("inmet_landing")

ano_inicial = 2015
ano_atual = dt.date.today().year


files = dbutils.fs.ls(VolumePath)


anos_existentes = []

for f in files:
    if not f.isDir() and Path(f.name).suffix.lower() == ".zip":
        try:
            ano_arquivo = int(Path(f.name).stem)

            if ano_inicial <= ano_arquivo <= ano_atual:
                anos_existentes.append(ano_arquivo)

        except ValueError:
            logger.warning(f"Nome de ZIP ignorado: {f.name}")

anos_existentes = sorted(set(anos_existentes))

logger.info(f"Anos com ZIP encontrado: {anos_existentes}")
logger.info(f"Intervalo de carga: {ano_inicial} a {ano_atual}")

## **Load**

In [0]:
anos_com_erro = []

with requests.Session() as session:

    for ano in range(ano_inicial, ano_atual + 1):

        caminho = Path(VolumePath) / f"{ano}.zip"
        caminho_temp = Path(VolumePath) / f"{ano}.zip.part"


        if ano != ano_atual and ano in anos_existentes:
            if z.is_zipfile(caminho):
                logger.info(f"ZIP histórico já existe e é válido: {caminho}")
                continue

            logger.warning(f"ZIP histórico inválido; será baixado novamente: {caminho}")

        try:
            url = f"{inmetPath}/{ano}.zip"

            logger.info(f"Iniciando download do ano {ano}")

            resposta = session.get(url, timeout=120)
            resposta.raise_for_status()

            if not resposta.content:
                raise ValueError("A resposta recebida está vazia.")


            caminho_temp.write_bytes(resposta.content)


            if not z.is_zipfile(caminho_temp):
                raise ValueError("O arquivo recebido não é um ZIP válido.")


            with z.ZipFile(caminho_temp, "r") as zip_ref:
                arquivo_corrompido = zip_ref.testzip()

                if arquivo_corrompido is not None:
                    raise ValueError(
                        f"Arquivo corrompido no ZIP: {arquivo_corrompido}"
                    )
    

            caminho_temp.replace(caminho)

            logger.info(f"Download concluído e validado: {caminho}")

        except Exception:
            logger.exception(f"Falha ao processar o ano {ano}")
            anos_com_erro.append(ano)

        finally:
            if caminho_temp.exists():
                caminho_temp.unlink()

if anos_com_erro:
    raise RuntimeError(
        f"Falha ao baixar ou validar os ZIPs dos anos: {anos_com_erro}"
    )

logger.info("Processamento da Landing do INMET concluído com sucesso.")